# Data Cleaning: From Messy Data to Analysis-Ready
**Oasis Infobyte SIP · Data Analytics · Level 1 · Task 3**

**Objective:** take a deliberately messy customer dataset and systematically transform it into a clean, analysis-ready dataset, documenting every decision.

**Dataset:** `messy_customer_data.csv` (1,300 rows, 11 columns: Customer_ID, Name, Gender, Age, Join_Date, City, Income, Purchase_Amount, Rating, Email, Is_Member). It has missing values, duplicates, inconsistent labels and formats, wrong data types, invalid values and outliers.

> The file included here is generated by `generate_data.py` so that every type of problem is present. To use a real messy dataset from Kaggle (e.g. search "data cleaning"), save it as `messy_customer_data.csv` and adapt the column names; then update the written observations to match your results.

**Cleaning plan:** (1) data quality report -> (2) standardise formats and labels -> (3) remove duplicates -> (4) fix invalid values -> (5) treat outliers -> (6) handle missing data -> (7) correct data types -> (8) before vs after summary -> (9) save cleaned CSV.

---
## 1. Load the data

In [ ]:
import os, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid", palette="deep")
plt.rcParams["figure.figsize"] = (10, 5)
os.makedirs("images", exist_ok=True)

raw = pd.read_csv("messy_customer_data.csv")
df = raw.copy()          # we clean `df` and keep `raw` untouched for before/after comparison
print("Shape:", raw.shape)
raw.head(10)

## 2. Data quality report (before cleaning)

In [ ]:
EMAIL_RE = r"^[\w.\-]+@[\w\-]+(\.[\w\-]+)+$"

def to_number(s):
    """Turn messy text like 'Rs. 55,000' or '1,200' into numbers; anything unparseable becomes NaN."""
    cleaned = s.astype(str).str.replace(r"(?i)rs\.?|,|\s", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

def parse_dates(s):
    """The column mixes 3 formats; try each one explicitly (safer than letting pandas guess)."""
    out = pd.to_datetime(s, format="%Y-%m-%d", errors="coerce")
    out = out.fillna(pd.to_datetime(s, format="%d/%m/%Y", errors="coerce"))
    out = out.fillna(pd.to_datetime(s, format="%b %d, %Y", errors="coerce"))
    return out

def count_invalid(d):
    age, inc, rat = to_number(d["Age"]), to_number(d["Income"]), to_number(d["Rating"])
    pur = to_number(d["Purchase_Amount"])
    mail = d["Email"].astype(str).str.strip().str.lower()
    bad_mail = d["Email"].notna() & ~mail.str.match(EMAIL_RE)
    return int(((age < 18) | (age > 100)).sum() + (inc < 0).sum() + (pur < 0).sum()
               + (rat.notna() & ~rat.isin([1, 2, 3, 4, 5])).sum() + bad_mail.sum())

def _is_str(s):
    return pd.api.types.is_string_dtype(s) and not isinstance(s.dtype, pd.CategoricalDtype)
def _is_cat(s):
    return isinstance(s.dtype, pd.CategoricalDtype)

EXPECTED = {  # the dtype each column SHOULD have
    "Customer_ID": _is_str, "Name": _is_str, "Gender": _is_cat,
    "Age": pd.api.types.is_integer_dtype, "Join_Date": pd.api.types.is_datetime64_any_dtype,
    "City": _is_cat, "Income": pd.api.types.is_float_dtype,
    "Purchase_Amount": pd.api.types.is_float_dtype, "Rating": pd.api.types.is_integer_dtype,
    "Email": _is_str, "Is_Member": pd.api.types.is_bool_dtype,
}

def summarise(d):
    return {
        "Row count": len(d),
        "Total null cells": int(d.isnull().sum().sum()),
        "Columns containing nulls": int((d.isnull().sum() > 0).sum()),
        "Exact duplicate rows": int(d.duplicated().sum()),
        "Duplicate Customer_IDs (case-insensitive)": int(d["Customer_ID"].astype(str).str.strip().str.upper().duplicated().sum()),
        "Invalid values (age/income/rating/email)": count_invalid(d),
        "Columns with correct dtype": f"{sum(f(d[c]) for c, f in EXPECTED.items())} of {len(EXPECTED)}",
    }

before_summary = summarise(raw)

In [ ]:
print("=== DATA QUALITY REPORT (raw data) ===\n")
print("Rows x columns:", raw.shape)
print("\n1) Null values per column")
nulls = pd.DataFrame({"nulls": raw.isnull().sum(), "% of rows": (raw.isnull().mean() * 100).round(1)})
print(nulls)

print("\n2) Duplicates")
print("   Exact duplicate rows:", raw.duplicated().sum())
print("   Duplicate Customer_IDs (ignoring case):", raw["Customer_ID"].str.upper().duplicated().sum())

print("\n3) Data types (current)")
print(raw.dtypes)

print("\n4) Value-range anomalies")
age_raw, inc_raw, rat_raw = to_number(raw["Age"]), to_number(raw["Income"]), to_number(raw["Rating"])
print("   Age outside 18-100:", int(((age_raw < 18) | (age_raw > 100)).sum()), "| range:", age_raw.min(), "to", age_raw.max())
print("   Income negative:", int((inc_raw < 0).sum()), "| above 1,000,000:", int((inc_raw > 1_000_000).sum()))
print("   Rating outside 1-5:", int((rat_raw.notna() & ~rat_raw.isin([1, 2, 3, 4, 5])).sum()))
bad_mail = raw["Email"].notna() & ~raw["Email"].astype(str).str.strip().str.lower().str.match(EMAIL_RE)
print("   Invalid email format:", int(bad_mail.sum()))

print("\n5) Inconsistent labels (distinct values)")
for c in ["Gender", "City", "Is_Member"]:
    print(f"   {c}: {raw[c].nunique()} distinct -> {sorted(raw[c].dropna().astype(str).unique())[:12]}...")
print("   Join_Date: mixes ISO (2023-02-10), DD/MM/YYYY and 'Sep 16, 2023' formats")

**Observation:** the raw data has 1,300 rows and **508 missing cells spread over 9 of 11 columns** (highest: Rating 6.8%, Income 5.8%, Age 5.5%). Other problems found:
- **Duplicates:** 60 exact duplicate rows, and 100 duplicate Customer_IDs once upper/lower case is ignored, which means 40 duplicates are hidden by formatting differences.
- **Wrong data types:** only 4 of 11 columns have the right type. Income is text (`"Rs. 55,000"`), Join_Date is text in three different formats, and Gender, City and Is_Member are plain strings.
- **Invalid values:** 6 impossible ages (from -5 to 999), 3 negative incomes, 9 incomes above 1,000,000, 4 ratings outside 1-5 and 9 malformed emails.
- **Inconsistent labels:** Gender has 12 spellings, City has 26 (case, spaces, aliases like Bombay/Bengaluru) and Is_Member has 10 (Yes/Y/True/1...).

## 3. Standardisation: formats, labels and data types of text, numbers and dates

In [ ]:
# --- Text columns: trim spaces and fix case
df["Customer_ID"] = df["Customer_ID"].str.strip().str.upper()
df["Name"] = df["Name"].str.strip().str.title()
df["Email"] = df["Email"].str.strip().str.lower()

# --- Gender: many spellings -> Male / Female
gender_map = {"m": "Male", "male": "Male", "f": "Female", "fem": "Female", "female": "Female"}
df["Gender"] = df["Gender"].str.strip().str.lower().map(gender_map)

# --- City: fix case/spacing and merge aliases (Bombay = Mumbai, etc.)
city_alias = {"bombay": "Mumbai", "new delhi": "Delhi", "bengaluru": "Bangalore"}
df["City"] = df["City"].str.strip().str.lower().replace(city_alias).str.title()

# --- Is_Member: Yes/Y/True/1 and No/N/False/0 -> True/False
member_map = {"yes": True, "y": True, "true": True, "1": True, "no": False, "n": False, "false": False, "0": False}
df["Is_Member"] = df["Is_Member"].str.strip().str.lower().map(member_map).astype("boolean")

# --- Numbers stored as text ("Rs. 55,000", "1,200") -> real numbers
for col in ["Age", "Income", "Purchase_Amount", "Rating"]:
    df[col] = to_number(df[col])

# --- Dates in 3 formats -> one datetime column
df["Join_Date"] = parse_dates(df["Join_Date"])

print("Gender :", sorted(df["Gender"].dropna().unique()))
print("City   :", sorted(df["City"].dropna().unique()))
print("Member :", df["Is_Member"].dropna().unique().tolist())
print("Dates parsed:", df["Join_Date"].notna().sum(), "of", raw["Join_Date"].notna().sum(), "non-null raw values")

**Decisions:**
- Names are trimmed and title-cased; IDs upper-cased; emails lower-cased.
- Gender spellings are mapped to `Male` / `Female` (12 spellings -> 2); City case/spacing is fixed and aliases are merged (`Bombay`->`Mumbai`, `New Delhi`->`Delhi`, `Bengaluru`->`Bangalore`), giving 26 -> 6 cities; Is_Member becomes True/False (10 -> 2 labels).
- Text like `Rs. 55,000` is converted to numbers; anything that cannot be read as a number becomes missing instead of crashing.
- The three date formats are parsed one by one with explicit formats (safer than letting pandas guess day/month order). All 1,261 non-missing dates were parsed successfully.

## 4. Duplicate removal

In [ ]:
rows_before = len(df)
df = df.drop_duplicates()
exact_removed = rows_before - len(df)

# Same customer appearing twice with slightly different data: keep the most complete record
df = (df.assign(_filled=df.notna().sum(axis=1))
        .sort_values(["Customer_ID", "_filled"], ascending=[True, False])
        .drop_duplicates(subset="Customer_ID", keep="first")
        .drop(columns="_filled")
        .sort_values("Customer_ID").reset_index(drop=True))
id_removed = rows_before - exact_removed - len(df)

print(f"Rows before: {rows_before:,}")
print(f"Exact duplicate rows removed: {exact_removed}")
print(f"Extra duplicate Customer_IDs removed: {id_removed}")
print(f"Rows after de-duplication: {len(df):,}  (unique customers: {df['Customer_ID'].nunique():,})")

**Observation:** **100 duplicate rows were removed** (1,300 -> 1,200 unique customers). Only 60 were visible in the raw data; the other 40 became exact duplicates *after* standardisation (they differed only by upper/lower case). This is why standardisation should come before de-duplication. The code also keeps the most complete record if the same Customer_ID ever appears with different data (none remained here).

## 5. Invalid values -> missing

In [ ]:
inv_age = int(((df["Age"] < 18) | (df["Age"] > 100)).sum())
inv_inc = int((df["Income"] < 0).sum())
inv_pur = int((df["Purchase_Amount"] < 0).sum())
inv_rat = int((df["Rating"].notna() & ~df["Rating"].isin([1, 2, 3, 4, 5])).sum())
bad_email_mask = df["Email"].notna() & ~df["Email"].str.match(EMAIL_RE)
inv_mail = int(bad_email_mask.sum())

df.loc[(df["Age"] < 18) | (df["Age"] > 100), "Age"] = np.nan
df.loc[df["Income"] < 0, "Income"] = np.nan
df.loc[df["Purchase_Amount"] < 0, "Purchase_Amount"] = np.nan
df.loc[~df["Rating"].isin([1, 2, 3, 4, 5]), "Rating"] = np.nan
df.loc[bad_email_mask, "Email"] = np.nan

print(f"Impossible ages set to missing: {inv_age} | negative incomes: {inv_inc} | negative purchases: {inv_pur}")
print(f"Ratings outside 1-5: {inv_rat} | invalid emails: {inv_mail}")

nulls_raw = raw.isnull().sum()
nulls_stage2 = df.isnull().sum()
pd.DataFrame({"raw nulls": nulls_raw, "after fixing invalid values": nulls_stage2})

**Decision:** impossible values are not "real" data, so they are converted to missing and handled together with the other missing values: ages outside 18-100 (6), negative incomes (3), ratings outside 1-5 (4) and malformed emails (8). The table shows that the *true* amount of missing data is higher than the raw null count suggests (for example Email 64 -> 68).

## 6. Outlier detection and treatment (IQR and Z-score)

In [ ]:
def iqr_bounds(s, k=1.5):
    q1, q3 = s.quantile(0.25), s.quantile(0.75)
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr

income_before_cap = df["Income"].copy()
purch_before_cap = df["Purchase_Amount"].copy()

# Income: IQR method (1.5 x IQR)
inc_lo, inc_hi = iqr_bounds(df["Income"].dropna())
n_inc_out = int(((df["Income"] < inc_lo) | (df["Income"] > inc_hi)).sum())
# Purchase_Amount: mild fence (1.5) and extreme fence (3 x IQR)
_, pur_hi = iqr_bounds(df["Purchase_Amount"].dropna())
_, pur_hi_extreme = iqr_bounds(df["Purchase_Amount"].dropna(), k=3)
n_pur_mild = int(((df["Purchase_Amount"] > pur_hi) & (df["Purchase_Amount"] <= pur_hi_extreme)).sum())
n_pur_extreme = int((df["Purchase_Amount"] > pur_hi_extreme).sum())
# Age: Z-score method
z_age = (df["Age"] - df["Age"].mean()) / df["Age"].std()
n_age_z = int((z_age.abs() > 3).sum())

print(f"Income        : IQR fences {max(inc_lo, 0):,.0f} to {inc_hi:,.0f} -> {n_inc_out} outliers")
print(f"Purchase_Amt  : mild fence {pur_hi:,.0f} ({n_pur_mild} mild outliers) | extreme fence {pur_hi_extreme:,.0f} ({n_pur_extreme} extreme)")
print(f"Age           : Z-score |z|>3 -> {n_age_z} outliers")

# Decisions
df["Income"] = df["Income"].clip(upper=inc_hi)                     # cap (winsorise)
df["Purchase_Amount"] = df["Purchase_Amount"].clip(upper=pur_hi_extreme)  # cap only extreme errors

fig, ax = plt.subplots(2, 2, figsize=(13, 6))
for i, (name, b, a) in enumerate([("Income", income_before_cap, df["Income"]),
                                  ("Purchase_Amount", purch_before_cap, df["Purchase_Amount"])]):
    sns.boxplot(x=b.dropna(), ax=ax[i, 0], color="#EA580C"); ax[i, 0].set_title(f"{name} - before outlier treatment")
    sns.boxplot(x=a.dropna(), ax=ax[i, 1], color="#16A34A"); ax[i, 1].set_title(f"{name} - after outlier treatment")
plt.tight_layout()
plt.savefig("images/02_outliers_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

**Decisions (one per column):**

| Column | Method | Result | Decision and reason |
|---|---|---|---|
| Income | IQR (1.5x) | upper fence 122,881; 42 outliers | **Cap** at the upper fence. Values such as 5,000,000 are obvious data-entry errors, and capping keeps the customer rows. |
| Purchase_Amount | IQR (1.5x and 3x) | 50 mild outliers above 1,386; 23 extreme above 2,081 | **Retain mild** outliers (real big spenders; purchase data is naturally right-skewed) and **cap only the extreme** ones above 2,081. |
| Age | Z-score (\|z\| > 3) | 0 outliers | **Retain**: after removing impossible ages, nothing unusual is left. |

The boxplots show the extreme Income and Purchase values squashing the box before treatment, and a readable distribution after.

## 7. Missing data handling

In [ ]:
miss_before = df.isnull().sum()
n_before_drop = len(df)

# 1) Join_Date: cannot invent a signup date -> delete those rows
df = df.dropna(subset=["Join_Date"]).copy()
n_date_dropped = n_before_drop - len(df)

# 2) Numeric columns: median (robust to skew/outliers)
age_med, income_med = df["Age"].median(), df["Income"].median()
purch_med, rating_med = df["Purchase_Amount"].median(), df["Rating"].median()
df["Age"] = df["Age"].fillna(age_med)
df["Income"] = df["Income"].fillna(income_med)
df["Purchase_Amount"] = df["Purchase_Amount"].fillna(purch_med)
df["Rating"] = df["Rating"].fillna(rating_med)

# 3) Categorical: City -> mode; Is_Member -> mode; Gender -> explicit "Unknown" label
city_mode = df["City"].mode()[0]
member_mode = bool(df["Is_Member"].mode()[0])
df["City"] = df["City"].fillna(city_mode)
df["Is_Member"] = df["Is_Member"].fillna(member_mode)
df["Gender"] = df["Gender"].fillna("Unknown")

# 4) Email: left missing on purpose (cannot be inferred)
print(f"Rows deleted (missing Join_Date): {n_date_dropped}")
print(f"Median Age {age_med:.0f} | Median Income {income_med:,.0f} | Median Purchase {purch_med:,.2f} | Median Rating {rating_med:.0f}")
print(f"City mode: {city_mode} | Is_Member mode: {member_mode}")
print("\nRemaining nulls:\n", df.isnull().sum()[df.isnull().sum() > 0])

**Strategy per column (and why):**

| Column | Strategy | Justification |
|---|---|---|
| Join_Date | **Delete rows** (36 rows) | A signup date cannot be guessed, and 3% of rows is a small loss. |
| Age | **Median** (37) | Robust to skew and to any leftover extreme values. |
| Income | **Median** (53,945) | Income is right-skewed; the mean would be pulled up by high earners. |
| Purchase_Amount | **Median** (400.83) | Spending is right-skewed, so the median is more typical than the mean. |
| Rating | **Median** (4) | Ordinal 1-5 scale; the median stays a valid rating. |
| City | **Mode** (Delhi) | Categorical; the most frequent value is the least risky fill (a limitation: it slightly over-represents Delhi). |
| Is_Member | **Mode** (True) | Categorical yes/no; same reasoning as City. |
| Gender | **"Unknown" label** (45 rows) | Guessing a person's gender is inappropriate; an explicit category is more honest. |
| Email | **Left missing** (68 rows) | An email cannot be inferred and this field is not needed for numeric analysis. |

## 8. Correct data types

In [ ]:
df["Customer_ID"] = df["Customer_ID"].astype("string")
df["Name"] = df["Name"].astype("string")
df["Email"] = df["Email"].astype("string")
df["Gender"] = df["Gender"].astype("category")
df["City"] = df["City"].astype("category")
df["Age"] = df["Age"].round().astype("Int64")
df["Rating"] = df["Rating"].round().astype("Int64")
df["Income"] = df["Income"].astype("float64")
df["Purchase_Amount"] = df["Purchase_Amount"].astype("float64")
df["Is_Member"] = df["Is_Member"].astype("boolean")
df = df.reset_index(drop=True)
pd.DataFrame({"raw dtype": raw.dtypes.astype(str), "clean dtype": df.dtypes.astype(str)})

**Observation:** all 11 columns now have the right type: IDs, names and emails as `string`; Gender and City as `category` (saves memory and prevents typos); Age and Rating as `Int64`; Income and Purchase_Amount as `float64`; Join_Date as `datetime64`; Is_Member as `boolean`. This fixes the problems found in the report (only 4 of 11 were correct before).

## 9. Before vs after summary

In [ ]:
after_summary = summarise(df)
summary_table = pd.DataFrame({"Before cleaning": before_summary, "After cleaning": after_summary})
summary_table

**Observation:**

| Metric | Before | After |
|---|---|---|
| Row count | 1,300 | 1,164 |
| Total null cells | 508 | 68 (all Email, left on purpose) |
| Columns containing nulls | 9 | 1 |
| Exact duplicate rows | 60 | 0 |
| Duplicate Customer_IDs | 100 | 0 |
| Invalid values | 22 | 0 |
| Columns with correct dtype | 4 of 11 | 11 of 11 |

We lost 136 rows in total (100 duplicates + 36 rows without a join date), about 10%, and in exchange every column is consistent, correctly typed and free of impossible values.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 5))
cmp = pd.DataFrame({"Raw": raw.isnull().sum(),
                    "After fixing invalid values": nulls_stage2,
                    "Final (cleaned)": df.isnull().sum()})
cmp.plot.barh(ax=ax[0], color=["#EA580C", "#F9AB00", "#16A34A"])
ax[0].set_title("Missing values per column"); ax[0].set_xlabel("Number of nulls")

labels = pd.DataFrame({
    "Before": [raw[c].nunique() for c in ["Gender", "City", "Is_Member"]],
    "After": [df[c].nunique() for c in ["Gender", "City", "Is_Member"]]},
    index=["Gender", "City", "Is_Member"])
labels.plot.bar(ax=ax[1], rot=0, color=["#EA580C", "#16A34A"])
ax[1].set_title("Distinct labels per column (fewer = more consistent)"); ax[1].set_ylabel("Distinct values")
for cont in ax[1].containers:
    ax[1].bar_label(cont)
plt.tight_layout()
plt.savefig("images/01_missing_and_labels_before_after.png", dpi=150, bbox_inches="tight")
plt.show()

**Observation:** the left chart shows missing values before cleaning, after invalid values were converted to missing, and in the final data (only Email remains). The right chart shows the label clean-up: Gender 12 -> 3 (Male, Female, Unknown), City 26 -> 6 and Is_Member 10 -> 2.

## 10. Save the cleaned dataset

In [ ]:
df.to_csv("cleaned_customer_data.csv", index=False)
print("Saved cleaned_customer_data.csv:", df.shape)
df.head(10)

## 11. Conclusion
- The raw data had problems of every common kind. The most valuable lesson: **order matters**. Standardising before de-duplicating revealed 40 hidden duplicates.
- Each decision (delete, impute, cap, retain, relabel) was justified instead of applied blindly, and the before/after table proves the result.
- The cleaned file `cleaned_customer_data.csv` (1,164 rows) is ready for analysis such as customer segmentation or spend modelling.
- **Limitations:** median/mode imputation reduces natural variability; capping changes extreme values; Email gaps remain by design.